# 07 · Des bases de données vers les fichiers

On récupère des données de PostgreSQL et de MongoDB (vus au TP « Sources de données »), on les enregistre en CSV, JSON et Parquet,
puis on compare les tailles, la relecture et ce qui est conservé.

> ⚠️ Il faut le fichier `.env` du TP « Sources de données » à la racine du projet (modèle : `.env.example`).
> La base est partagée par toute la promo : on ne fait que **lire**, avec des requêtes **légères**.

## 1. Se connecter aux deux bases

Les identifiants sont lus dans `.env` et ne sont jamais affichés.

In [1]:
import os
from pathlib import Path
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient
from sqlalchemy import URL, create_engine

In [2]:
BDD = Path("../output/bdd")
BDD.mkdir(parents=True, exist_ok=True)

def taille_ko(nom):
    """Taille d'un fichier du dossier output/bdd, en Ko."""
    return round((BDD / nom).stat().st_size / 1_000, 1)

In [3]:
load_dotenv("../.env")
for nom in ["PG_HOST", "PG_PORT", "PG_DATABASE", "PG_USER", "PG_PASSWORD", "MONGO_URI", "MONGO_DB"]:
    if not os.getenv(nom):
        raise ValueError(f"{nom} manquant : copiez le .env du TP Sources de données à la racine du projet.")
print("Identifiants chargés (non affichés).")

Identifiants chargés (non affichés).


Même connexion qu'au TP « Sources de données » : le *pooler* Supabase (port 6543) impose `prepare_threshold=None`.
Ces deux cellules n'affichent rien : la connexion ne s'ouvre qu'à la première requête.

In [4]:
url = URL.create(
    "postgresql+psycopg",
    username=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=int(os.getenv("PG_PORT")),
    database=os.getenv("PG_DATABASE"),
)
engine = create_engine(url, connect_args={"prepare_threshold": None})

In [5]:
client = MongoClient(os.getenv("MONGO_URI"), serverSelectionTimeoutMS=10000)
db = client[os.getenv("MONGO_DB")]

## 2. PostgreSQL : la table `eleves`

`pd.read_sql` envoie la requête et range le résultat dans un DataFrame. `parse_dates` transforme les colonnes de dates en vraies dates.

In [6]:
dates = ["date_naissance", "date_inscription"]
eleves = pd.read_sql("SELECT * FROM eleves", engine, parse_dates=dates)
print(eleves.shape)
eleves[["id", "nom", "prenom", "date_naissance", "nationalite"]].head(3)

(5549, 13)


,id,nom,prenom,date_naissance,nationalite
0,1,Noël,Léon,2004-10-12,Sénégalaise
1,2,Lacroix,Frédérique,2004-01-11,Française
2,3,Dumas,Josette,2005-11-09,Sénégalaise


5 549 élèves et 13 colonnes : une petite table, qu'on peut lire en entier sans gêner le reste de la promo.

## 3. Enregistrer en CSV et en Parquet

In [7]:
eleves.to_csv(BDD / "eleves.csv", index=False)
eleves.to_parquet(BDD / "eleves.parquet", index=False)
print("CSV     :", taille_ko("eleves.csv"), "Ko")
print("Parquet :", taille_ko("eleves.parquet"), "Ko")

CSV     : 923.3 Ko
Parquet : 357.2 Ko


Le Parquet est plus petit que le CSV, même sur une petite table : il est compressé et range les valeurs colonne par colonne.

## 4. Relire : vitesse et types

On relit les deux fichiers en mesurant la durée avec `%%time` (*Wall time* = durée réelle écoulée).

In [8]:
%%time
eleves_csv = pd.read_csv(BDD / "eleves.csv")

CPU times: user 42.7 ms, sys: 5.87 ms, total: 48.6 ms
Wall time: 49.2 ms


In [9]:
%%time
eleves_parquet = pd.read_parquet(BDD / "eleves.parquet")

CPU times: user 22.4 ms, sys: 289 ms, total: 311 ms
Wall time: 112 ms


Les deux lectures ne prennent que quelques millisecondes : sur une si petite table, la vitesse ne départage pas les formats (voir le notebook 06 pour un mois entier de taxis).
La vraie différence est dans les types.

In [10]:
pd.DataFrame({"base": eleves.dtypes, "csv": eleves_csv.dtypes, "parquet": eleves_parquet.dtypes})

,base,csv,parquet
id,int64,int64,int64
nom,str,str,str
prenom,str,str,str
date_naissance,datetime64[s],str,datetime64[ms]
genre,str,str,str
adresse,str,str,str
ville_id,int64,int64,int64
email_parent,str,str,str
telephone_parent,str,str,str
nom_parent,str,str,str


Dans le CSV relu, les dates sont redevenues du texte (`str`) : un CSV ne contient que des caractères.
Le Parquet, lui, rend bien des dates (`datetime64`).

## 5. MongoDB : 5 000 courses

On prend 5 000 courses (`limit`), sans les identifiants (`_id`, `userId`…) : ce sont des `ObjectId`, un type propre à MongoDB que les fichiers ne connaissent pas.

In [11]:
projection = {"_id": 0, "userId": 0, "driverId": 0, "vehicleId": 0}
documents = list(db.trips.find({}, projection).limit(5000))
print(len(documents), "documents")
pprint(documents[0])

5000 documents
{'acceptedAt': datetime.datetime(2025, 11, 21, 16, 22, 19, 512000),
 'amount': 2779.3340900417015,
 'currency': 'SLL',
 'destination': {'coordinates': [114.339762, -54.625561], 'type': 'Point'},
 'endedAt': datetime.datetime(2025, 11, 21, 17, 6, 13, 512000),
 'fare': {'amount': 2779.3340900417015,
          'breakdown': {'base': 757.9614589030713,
                        'distance': 2735.1317931362128,
                        'time': 809.1405860269375},
          'currency': 'SLL'},
 'origin': {'coordinates': [78.717465, 26.849791], 'type': 'Point'},
 'requestedAt': datetime.datetime(2025, 11, 21, 16, 16, 16, 512000),
 'serviceDetails': {'packageWeightKg': None, 'rentalDurationMins': None},
 'serviceType': 'ride',
 'startedAt': datetime.datetime(2025, 11, 21, 16, 24, 52, 512000),
 'status': 'completed'}


Un document est un dictionnaire Python : `fare` contient un sous-document (`breakdown`), `origin` une liste `coordinates`, et les dates sont des `datetime`.
Ce n'est pas un tableau plat.

## 6. Enregistrer en JSON, Parquet et CSV

Dans un DataFrame, chaque sous-document (`fare`, `origin`…) devient une colonne de dictionnaires.

In [12]:
trips = pd.DataFrame(documents)
trips[["requestedAt", "fare", "origin"]].head(3)

,requestedAt,fare,origin
0,2025-11-21 16:16:16.512,"{'amount': 2779.3340900417015, 'currency': 'SL...","{'type': 'Point', 'coordinates': [78.717465, 2..."
1,2024-06-27 16:23:27.512,"{'amount': 2776.63264776803, 'currency': 'GHS'...","{'type': 'Point', 'coordinates': [-24.967116, ..."
2,2025-09-18 00:00:48.512,"{'amount': 2581.6113322542374, 'currency': 'XA...","{'type': 'Point', 'coordinates': [-91.158122, ..."


**JSON**, **JSON Lines** et **Parquet** savent stocker ces dictionnaires et ces listes : on enregistre le DataFrame tel quel.
JSON n'a pas de type date : `date_format="iso"` les écrit en texte (code de la slide 13).

In [13]:
trips.to_json(BDD / "trips.json", orient="records", date_format="iso")
trips.to_json(BDD / "trips.jsonl", orient="records", lines=True, date_format="iso")
trips.to_parquet(BDD / "trips.parquet", index=False)

with open(BDD / "trips.jsonl", encoding="utf-8") as f:
    print(f.readline(), end="")

{"serviceType":"ride","origin":{"type":"Point","coordinates":[78.717465,26.849791]},"destination":{"type":"Point","coordinates":[114.339762,-54.625561]},"requestedAt":"2025-11-21T16:16:16.512","acceptedAt":"2025-11-21T16:22:19.512","startedAt":"2025-11-21T16:24:52.512","endedAt":"2025-11-21T17:06:13.512","status":"completed","currency":"SLL","amount":2779.3340900417,"fare":{"amount":2779.3340900417,"currency":"SLL","breakdown":{"base":757.9614589031,"distance":2735.1317931362,"time":809.1405860269}},"serviceDetails":{"rentalDurationMins":null,"packageWeightKg":null}}


Une ligne du JSON Lines = un document complet, avec ses objets `{…}` et ses listes `[…]`. La date `requestedAt` y est du texte.

Le **CSV** est plat : il faut d'abord aplatir avec `pd.json_normalize` (slide 14). Chaque sous-champ devient une colonne.

In [14]:
plat = pd.json_normalize(documents, sep="_")
plat.to_csv(BDD / "trips_plat.csv", index=False)
plat[["requestedAt", "fare_amount", "fare_breakdown_base", "origin_coordinates"]].head(3)

,requestedAt,fare_amount,fare_breakdown_base,origin_coordinates
0,2025-11-21 16:16:16.512,2779.334090,757.961459,"[78.717465, 26.849791]"
1,2024-06-27 16:23:27.512,2776.632648,796.436667,"[-24.967116, 59.6519015]"
2,2025-09-18 00:00:48.512,2581.611332,451.077560,"[-91.158122, -48.3549915]"


`fare.breakdown.base` est devenu la colonne `fare_breakdown_base`. Mais la liste `origin_coordinates` ne s'aplatit pas : le CSV l'écrit comme du texte.

## 7. Relire : qu'est-ce qui revient ?

In [15]:
trips_csv = pd.read_csv(BDD / "trips_plat.csv")
trips_parquet = pd.read_parquet(BDD / "trips.parquet")
print("CSV     :", type(trips_csv["origin_coordinates"][0]))
print("Parquet :", type(trips_parquet["fare"][0]))
trips_parquet["fare"][0]

CSV     : <class 'str'>
Parquet : <class 'dict'>


{'amount': 2779.3340900417015,
 'currency': 'SLL',
 'breakdown': {'base': 757.9614589030713,
  'distance': 2735.1317931362128,
  'time': 809.1405860269375}}

Dans le CSV, les coordonnées ne sont plus qu'un texte `"[…, …]"`. Le Parquet rend `fare` comme un vrai dictionnaire, sous-document `breakdown` compris.

## 8. Le comparatif

In [16]:
comparatif = pd.DataFrame({
    "source": ["PostgreSQL", "PostgreSQL", "MongoDB", "MongoDB", "MongoDB", "MongoDB"],
    "fichier": ["eleves.csv", "eleves.parquet", "trips.json", "trips.jsonl", "trips_plat.csv", "trips.parquet"],
    "conservé": ["valeurs, sans les types", "valeurs et types", "imbrication, dates en texte",
                 "imbrication, dates en texte", "ni types ni imbrication", "types et imbrication"],
})
comparatif["taille_ko"] = comparatif["fichier"].map(taille_ko)
comparatif

,source,fichier,conservé,taille_ko
0,PostgreSQL,eleves.csv,"valeurs, sans les types",923.3
1,PostgreSQL,eleves.parquet,valeurs et types,357.2
2,MongoDB,trips.json,"imbrication, dates en texte",2897.6
3,MongoDB,trips.jsonl,"imbrication, dates en texte",2897.6
4,MongoDB,trips_plat.csv,ni types ni imbrication,1430.4
5,MongoDB,trips.parquet,types et imbrication,647.7


Pour les courses, JSON et JSON Lines sont les plus gros : les noms des champs sont répétés dans chaque document.
Parquet est le plus petit et le seul à tout garder : les types **et** l'imbrication.

## Exercices

### ✍️ Exercice 1 — Les pays en JSON Lines et en Parquet

Exportez les 19 pays de la collection `countries` (champs `name`, `isoCode`, `currency`) en JSON Lines avec les accents lisibles (`force_ascii=False`), puis en Parquet. Comparez les tailles.

In [17]:
# ✅ Solution
projection = {"_id": 0, "name": 1, "isoCode": 1, "currency": 1}
pays = pd.DataFrame(list(db.countries.find({}, projection)))
pays.to_json(BDD / "exo07_pays.jsonl", orient="records", lines=True, force_ascii=False)
pays.to_parquet(BDD / "exo07_pays.parquet", index=False)
print("JSON Lines :", taille_ko("exo07_pays.jsonl"), "Ko")
print("Parquet    :", taille_ko("exo07_pays.parquet"), "Ko")

JSON Lines : 1.0 Ko
Parquet    : 2.5 Ko


Sur 19 petits documents, le Parquet est plus gros : ses métadonnées (schéma, statistiques) pèsent plus que les données. Parquet est fait pour les gros volumes.

### ✍️ Exercice 2 — Les élèves béninois pour Excel

Exportez les élèves de nationalité « Béninoise » en CSV pour un Excel français (`sep=";"`, `encoding="utf-8-sig"`), puis relisez le fichier.

In [18]:
# ✅ Solution
beninois = eleves[eleves["nationalite"] == "Béninoise"]
beninois.to_csv(BDD / "exo07_eleves_beninois.csv", index=False, sep=";", encoding="utf-8-sig")
relu = pd.read_csv(BDD / "exo07_eleves_beninois.csv", sep=";", encoding="utf-8-sig")
relu["nationalite"].value_counts()

nationalite
Béninoise    1907
Name: count, dtype: int64

Une seule nationalité, accents intacts. `utf-8-sig` ajoute en tête du fichier une marque invisible (le *BOM*) qui dit à Excel de lire en UTF-8.

## Fermer les connexions

La base est partagée : on rend les connexions en partant.

In [19]:
engine.dispose()
client.close()

## À retenir

- `pd.read_sql` (PostgreSQL) et `list(db.collection.find(...))` (MongoDB) suffisent pour sortir des données d'une base.
- Une table SQL est plate : CSV ou Parquet. Seul Parquet garde les types (les dates restent des dates).
- Un document MongoDB est imbriqué : JSON Lines ou Parquet le gardent tel quel ; pour un CSV, il faut aplatir (`json_normalize`).
- JSON n'a pas de type date : `date_format="iso"` les écrit en texte.
- Les identifiants restent dans `.env`, jamais dans le notebook.